In [2]:
# -*- coding: utf-8 -*-
"""
Helpdesk Process Predictor - Complete version
Predictive model for next activity prediction on helpdesk dataset.
"""
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, classification_report
import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Dropout, Concatenate
import pickle
import os

class HelpdeskPredictor:
    def __init__(self, max_seq_length=10, embedding_dim=50):
        self.max_seq_length = max_seq_length
        self.embedding_dim = embedding_dim
        self.model = None
        self.activity_encoder = LabelEncoder()
        self.feature_encoders = {}
        
    def prepare_data(self, file_path):
        """Load and prepare the data"""
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"File not found: {file_path}")
            
        df = pd.read_csv(file_path)
        df['Complete Timestamp'] = pd.to_datetime(df['Complete Timestamp'])
        df = df.sort_values(['Case ID', 'Complete Timestamp'])

        # Activity encoding
        df['Activity_encoded'] = self.activity_encoder.fit_transform(df['Activity']).astype(np.int32)
        
        # Feature encoding
        categorical_features = ['seriousness', 'service_level', 'service_type', 'workgroup']
        for feature in categorical_features:
            encoder = LabelEncoder()
            df[f'{feature}_encoded'] = encoder.fit_transform(df[feature]).astype(np.int32)
            self.feature_encoders[feature] = encoder
            
        print(f"\nDataset Statistics:")
        print(f"Total events: {len(df)}")
        print(f"Unique cases: {df['Case ID'].nunique()}")
        print(f"Unique activities: {df['Activity'].nunique()}")
        print("\nActivity distribution:")
        print(df['Activity'].value_counts())
            
        return df
        
    def create_sequences(self, df):
        """Create sequences for training"""
        sequences = []
        next_activities = []
        features = []
        case_ids = []
        
        encoded_features = ['seriousness_encoded', 'service_level_encoded', 
                          'service_type_encoded', 'workgroup_encoded']

        for case_id in df['Case ID'].unique():
            case_df = df[df['Case ID'] == case_id]
            activities = case_df['Activity_encoded'].values.astype(np.int32)
            
            for i in range(1, len(activities)):
                seq = activities[max(0, i-self.max_seq_length):i]
                sequences.append(seq)
                next_activities.append(activities[i])
                feat = case_df.iloc[i-1][encoded_features].values.astype(np.float32)
                features.append(feat)
                case_ids.append(case_id)

        X_seq = pad_sequences(sequences, maxlen=self.max_seq_length, 
                            padding='pre', dtype='int32')
        X_feat = np.array(features, dtype='float32')
        y = np.array(next_activities, dtype='int32')
        
        return X_seq, X_feat, y, case_ids

    def build_model(self, n_activities, n_features):
        """Build the model architecture"""
        # Sequence input
        seq_input = Input(shape=(self.max_seq_length,), dtype='int32', name='sequence_input')
        x = Embedding(input_dim=n_activities, output_dim=self.embedding_dim)(seq_input)
        x = LSTM(100)(x)
        
        # Feature input
        feat_input = Input(shape=(n_features,), dtype='float32', name='feature_input')
        x = Concatenate()([x, feat_input])
        
        # Dense layers
        x = Dense(100, activation='relu')(x)
        x = Dropout(0.2)(x)
        x = Dense(50, activation='relu')(x)
        x = Dropout(0.2)(x)
        
        # Output layer
        output = Dense(n_activities, activation='softmax')(x)
        
        # Create model
        model = Model(inputs=[seq_input, feat_input], outputs=output)
        model.compile(optimizer='adam',
                     loss='sparse_categorical_crossentropy',
                     metrics=['accuracy'])
        
        self.model = model
        return model

    def train_and_evaluate(self, X_seq, X_feat, y, epochs=50, batch_size=32, validation_split=0.2):
        """Train the model and provide detailed evaluation"""
        # Split the data
        X_seq_train, X_seq_test, X_feat_train, X_feat_test, y_train, y_test = train_test_split(
            X_seq, X_feat, y, test_size=0.2, random_state=42
        )
        
        if self.model is None:
            n_activities = len(self.activity_encoder.classes_)
            n_features = X_feat.shape[1]
            self.build_model(n_activities, n_features)
        
        print("\nModel Architecture:")
        self.model.summary()
            
        history = self.model.fit(
            [X_seq_train, X_feat_train],
            y_train,
            epochs=epochs,
            batch_size=batch_size,
            validation_split=validation_split,
            verbose=1
        )
        
        # Evaluate on test set
        y_pred = self.model.predict([X_seq_test, X_feat_test])
        y_pred_classes = np.argmax(y_pred, axis=1)
        
        # Convert encoded activities back to names
        activity_names = self.activity_encoder.classes_
        y_test_names = [activity_names[i] for i in y_test]
        y_pred_names = [activity_names[i] for i in y_pred_classes]
        
        # Detailed evaluation
        print("\nDetailed Model Evaluation:")
        print("\nClassification Report:")
        print(classification_report(y_test_names, y_pred_names, zero_division=1))
        
        # Calculate and display metrics per activity
        print("\nDetailed metrics per activity:")
        unique_activities = np.unique(y_test_names)
        
        metrics_df = pd.DataFrame(columns=['Activity', 'Support', 'Accuracy', 'Precision', 'Recall', 'F1'])
        
        for activity in unique_activities:
            true_activity = np.array(y_test_names) == activity
            pred_activity = np.array(y_pred_names) == activity
            
            support = np.sum(true_activity)
            if support > 0:
                accuracy = np.mean(true_activity == pred_activity)
                precision = np.sum(true_activity & pred_activity) / (np.sum(pred_activity) + 1e-10)
                recall = np.sum(true_activity & pred_activity) / (np.sum(true_activity) + 1e-10)
                f1 = 2 * (precision * recall) / (precision + recall + 1e-10)
                
                metrics_df = pd.concat([metrics_df, pd.DataFrame({
                    'Activity': [activity],
                    'Support': [support],
                    'Accuracy': [accuracy],
                    'Precision': [precision],
                    'Recall': [recall],
                    'F1': [f1]
                })], ignore_index=True)
        
        # Sort by support
        metrics_df = metrics_df.sort_values('Support', ascending=False)
        
        # Format and display
        pd.set_option('display.float_format', lambda x: '%.3f' % x)
        print("\nMetrics by activity (sorted by frequency):")
        print(metrics_df.to_string(index=False))
        
        return history, metrics_df

    def predict_next(self, sequence, features):
        """Predict next activity with probability"""
        if not isinstance(sequence, list):
            sequence = [sequence]
            
        # Convert sequence to encoded form
        seq_encoded = self.activity_encoder.transform(sequence)
        X_seq = pad_sequences([seq_encoded], maxlen=self.max_seq_length, 
                            padding='pre', dtype='int32')
        X_feat = np.array([features], dtype='float32')
        
        # Get predictions
        pred_probs = self.model.predict([X_seq, X_feat])[0]
        
        # Get top 3 predictions
        top_indices = pred_probs.argsort()[-3:][::-1]
        predictions = []
        for idx in top_indices:
            activity = self.activity_encoder.inverse_transform([idx])[0]
            probability = pred_probs[idx]
            predictions.append((activity, probability))
        
        return predictions

    def save_model(self, directory):
        """Save model and parameters"""
        if not os.path.exists(directory):
            os.makedirs(directory)
            
        # Save model in new format
        model_path = os.path.join(directory, 'helpdesk_model.keras')
        self.model.save(model_path)
        
        # Save state
        state = {
            'activity_encoder': self.activity_encoder,
            'feature_encoders': self.feature_encoders,
            'max_seq_length': self.max_seq_length,
            'embedding_dim': self.embedding_dim
        }
        state_path = os.path.join(directory, 'model_state.pkl')
        with open(state_path, 'wb') as f:
            pickle.dump(state, f)
            
        print(f"\nModel and state saved in: {directory}")

    @classmethod
    def load_model(cls, directory):
        """Load saved model and parameters"""
        with open(os.path.join(directory, 'model_state.pkl'), 'rb') as f:
            state = pickle.load(f)
            
        instance = cls(
            max_seq_length=state['max_seq_length'],
            embedding_dim=state['embedding_dim']
        )
        
        instance.activity_encoder = state['activity_encoder']
        instance.feature_encoders = state['feature_encoders']
        instance.model = load_model(os.path.join(directory, 'helpdesk_model.keras'))
        
        return instance

def analyze_real_examples(predictor, df):
    """Analyze real examples from the dataset"""
    print("\nReal Examples Analysis:")
    print("======================")
    
    # Get random cases
    random_cases = np.random.choice(df['Case ID'].unique(), 3, replace=False)
    
    for case_id in random_cases:
        case_df = df[df['Case ID'] == case_id].copy()
        case_df = case_df.sort_values('Complete Timestamp')
        
        print(f"\nCase ID: {case_id}")
        activities = case_df['Activity'].values
        
        # Analyze sequence points
        for i in range(2, len(activities)):
            current_sequence = activities[max(0, i-3):i]
            actual_next = activities[i]
            
            # Use actual features from the dataset
            feature_values = case_df.iloc[i-1][[f'{feat}_encoded' for feat in 
                ['seriousness', 'service_level', 'service_type', 'workgroup']]].values
            
            # Get predictions
            predictions = predictor.predict_next(current_sequence, feature_values)
            
            print(f"\nCurrent sequence: {' -> '.join(current_sequence)}")
            print(f"Actual next activity: {actual_next}")
            print("Top 3 predictions:")
            for act, prob in predictions:
                print(f"- {act}: {prob:.2f}")
            print("-" * 50)

def test_specific_sequences(predictor):
    """Test model with specific sequences"""
    print("\nTesting Specific Sequences:")
    print("==========================")
    
    test_cases = [
        (['Insert ticket', 'Assign seriousness'], "New ticket flow"),
        (['Take in charge ticket', 'Wait'], "Waiting flow"),
        (['Create SW anomaly', 'Wait', 'Resolve ticket'], "Software anomaly flow")
    ]
    
    # Use median values for features as default
    default_features = np.array([1, 1, 1, 1], dtype=np.float32)
    
    for sequence, desc in test_cases:
        print(f"\nTesting {desc}:")
        print(f"Sequence: {' -> '.join(sequence)}")
        
        predictions = predictor.predict_next(sequence, default_features)
        print("Predictions:")
        for activity, prob in predictions:
            print(f"- {activity}: {prob:.2f}")
        print("-" * 50)

def main():
    """Main execution function"""
    try:
        # Set up file paths
        current_dir = os.getcwd()
        data_path = os.path.join(current_dir, 'dataset', 'HelpDesk', 'finale.csv')
        model_save_dir = os.path.join(current_dir, 'models')
        
        # Initialize predictor
        predictor = HelpdeskPredictor()
        
        # Check if data file exists
        if not os.path.exists(data_path):
            raise FileNotFoundError(f"Data file not found at: {data_path}")
        
        print(f"Loading data from {data_path}...")
        df = predictor.prepare_data(data_path)
        
        print("\nPreparing sequences...")
        X_seq, X_feat, y, case_ids = predictor.create_sequences(df)
        
        print(f"\nTraining Data Shape:")
        print(f"X_seq shape: {X_seq.shape}")
        print(f"X_feat shape: {X_feat.shape}")
        print(f"y shape: {y.shape}")
        
        print("\nTraining model...")
        history, metrics_df = predictor.train_and_evaluate(X_seq, X_feat, y, epochs=50)
        
        # Create models directory if it doesn't exist
        if not os.path.exists(model_save_dir):
            os.makedirs(model_save_dir)
        
        # Save model
        predictor.save_model(model_save_dir)
        
        # Save metrics to CSV
        metrics_path = os.path.join(model_save_dir, 'activity_metrics.csv')
        metrics_df.to_csv(metrics_path, index=False)
        print(f"\nMetrics saved to: {metrics_path}")
        
        # Analyze examples
        analyze_real_examples(predictor, df)
        
        # Test specific sequences
        test_specific_sequences(predictor)
        
    except Exception as e:
        print(f"Error: {str(e)}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

Loading data from /Users/ivan/Desktop/MITLxPPM/MITLxPPM/dataset/HelpDesk/finale.csv...

Dataset Statistics:
Total events: 21348
Unique cases: 4580
Unique activities: 14

Activity distribution:
Activity
Take in charge ticket    5060
Resolve ticket           4983
Assign seriousness       4938
Closed                   4574
Wait                     1463
Require upgrade           119
Insert ticket             118
Create SW anomaly          67
Resolve SW anomaly         13
Schedule intervention       5
VERIFIED                    3
RESOLVED                    2
INVALID                     2
DUPLICATE                   1
Name: count, dtype: int64

Preparing sequences...

Training Data Shape:
X_seq shape: (16768, 10)
X_feat shape: (16768, 4)
y shape: (16768,)

Training model...

Model Architecture:


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ sequence_input      │ (None, 10)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 10, 50)    │        700 │ sequence_input[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 100)       │     60,400 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ feature_input       │ (None, 4)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 104)       │          0 │ lstm[0][0],       │
│ (Concatenate)       │                   │            │ feature_input[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 100)       │     10,500 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 100)       │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 50)        │      5,050 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 50)        │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 14)        │        714 │ dropout_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 77,364 (302.20 KB)

 Trainable params: 77,364 (302.20 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
336/336 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.5033 - loss: 1.4273 - val_accuracy: 0.7894 - val_loss: 0.6500
Epoch 2/50
336/336 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7835 - loss: 0.6900 - val_accuracy: 0.7905 - val_loss: 0.6364
Epoch 3/50
336/336 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7769 - loss: 0.6827 - val_accuracy: 0.7909 - val_loss: 0.6376
Epoch 4/50
336/336 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7904 - loss: 0.6579 - val_accuracy: 0.7976 - val_loss: 0.6201
Epoch 5/50
336/336 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7937 - loss: 0.6458 - val_accuracy: 0.7980 - val_loss: 0.6158
Epoch 6/50
336/336 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7976 - loss: 0.6308 - val_accuracy: 0.7950 - val_loss: 0.6197
Epoch 7/50
336/336 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7872 - loss: 0.6497 - val_accuracy: 0.7987 - val_loss: 0.6164
Epoch 8/50
336/336 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7907 - loss: 0.6299 - val_accuracy: 0.

/var/folders/9j/10v9ngt92ts7snck5_jqsz1m0000gn/T/ipykernel_28333/1387982654.py:167: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  metrics_df = pd.concat([metrics_df, pd.DataFrame({
Traceback (most recent call last):
  File "/var/folders/9j/10v9ngt92ts7snck5_jqsz1m0000gn/T/ipykernel_28333/1387982654.py", line 349, in main
    analyze_real_examples(predictor, df)
  File "/var/folders/9j/10v9ngt92ts7snck5_jqsz1m0000gn/T/ipykernel_28333/1387982654.py", line 274, in analyze_real_examples
    predictions = predictor.predict_next(current_sequence, feature_values)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/var/folders/9j/10v9ngt92ts7snck5_jqsz1m0000gn/T/ipykernel_28333/1387982654.py", line 192, in predict_ne